# Triumph dashboard video reader

Upload one video and download a full-frame MP4 labeled with the predicted dashboard, LCD, needle, speed and RPM. The RPM text is larger. This release uses three selected models; gear and odometer reading are not included.

**No Google Drive or OAuth sign-in is required.** Code and weights come from public GitHub release `v0.1.0`; model files are checked against SHA256 hashes. Colab itself requires your normal Google sign-in.

This is an experiment for this Triumph dashboard. Camera angles, glare and unfamiliar recordings can fail. Unavailable readings and their reasons are logged instead of inventing values. Detection availability is not accuracy.

Choose **Runtime → Change runtime type → T4 GPU** for faster inference, or choose **CPU** and explicitly select `cpu` below. Source/playback FPS and measured processing FPS are different. The full video is processed by default, with no frame skipping or output crop. Output is silent; variable-frame-rate input uses constant-rate playback and retains original timestamps in the logs.


In [ ]:
#@title 1. Install the public release
import sys, subprocess, shutil
from pathlib import Path
import google.colab  # This notebook performs neural inference only inside Colab.

CODE_REF = "v0.1.0"
REPOSITORY = "https://github.com/sainadh812/triumph-dashboard-reader.git"
CHECKOUT = Path("/content/triumph_dashboard_reader_public")
if not CHECKOUT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", CODE_REF, REPOSITORY, str(CHECKOUT)], check=True)
else:
    origin = subprocess.check_output(["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True).strip()
    if origin != REPOSITORY:
        raise RuntimeError("Existing checkout belongs to another project. Use a fresh Colab runtime.")
    head = subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True).strip()
    release = subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", CODE_REF + "^{commit}"], text=True).strip()
    if head != release:
        raise RuntimeError("Existing checkout is not the selected release. Use a fresh Colab runtime.")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(CHECKOUT)], check=True)
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
print("Code commit:", subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True).strip())
print("Setup complete. No Drive mount or private credentials are used.")


In [ ]:
#@title 2. Choose device and video window
DEVICE = "cuda" #@param ["cuda", "cpu"]
GPU_BATCH_SIZE = 16 #@param {type:"integer"}
CPU_BATCH_SIZE = 4 #@param {type:"integer"}
CPU_THREADS = 2 #@param {type:"integer"}
LAST_SECONDS = 0 #@param {type:"number"}
# LAST_SECONDS=0 processes the entire video; e.g. 120 processes its final 2 minutes.
if LAST_SECONDS < 0:
    raise ValueError("LAST_SECONDS must be zero or positive")
BATCH_SIZE = GPU_BATCH_SIZE if DEVICE == "cuda" else CPU_BATCH_SIZE
print("Device:", DEVICE, "batch:", BATCH_SIZE, "window:", "full video" if LAST_SECONDS == 0 else f"last {LAST_SECONDS}s")


In [ ]:
#@title 3. Browse and upload one video
import os
from google.colab import files
UPLOAD_DIR = Path("/content/triumph_uploaded_video")
UPLOAD_DIR.mkdir(exist_ok=True)
previous_directory = Path.cwd()
try:
    os.chdir(UPLOAD_DIR)
    uploaded = files.upload()
finally:
    os.chdir(previous_directory)
if len(uploaded) != 1:
    raise ValueError("Please select exactly one video and run this upload cell again.")
VIDEO_PATH = UPLOAD_DIR / Path(next(iter(uploaded))).name
if not VIDEO_PATH.is_file():
    raise FileNotFoundError("The browser upload did not finish.")
del uploaded
print("Ready:", VIDEO_PATH.name, "bytes:", VIDEO_PATH.stat().st_size)


In [ ]:
#@title 4. Predict dashboard, LCD, speed and RPM
from datetime import datetime, timezone
from triumph_dashboard_reader.run import run_video
OUTPUT_DIR = Path("/content/triumph_video_results") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%f")
result = run_video(VIDEO_PATH, OUTPUT_DIR, device=DEVICE, batch_size=BATCH_SIZE,
                   cpu_threads=CPU_THREADS, last_seconds=None if LAST_SECONDS == 0 else LAST_SECONDS)
report = result["metrics"]
print("Video:", result["video"])
print("Total video processing seconds:", round(report["timing"]["total_processing_seconds"], 2))
print("Model loading seconds:", round(report["model_load_seconds"], 2))
print("Source FPS:", round(report["source_fps"], 2))
print("Prediction pipeline FPS:", round(report["prediction_pipeline_fps"], 2))
print("Full processing FPS (decode + labels + encode + verification):", round(report["end_to_end_processing_fps"], 2))
print("These rates include conditional model execution; see coverage and unavailable reasons above.")


In [ ]:
#@title 5. Download the labeled video and complete results
from google.colab import files
files.download(result["video"])
files.download(result["archive"])
print("The ZIP contains the same MP4, JSON report, per-frame CSV and raw prediction JSONL.")
print("Wait until both browser downloads finish. Keep the runtime connected until your files are safe.")


### Release the runtime after saving
The runtime's files are temporary. Changing hardware or disconnecting/deleting the runtime can remove them. The download call starts a browser transfer; it cannot prove that the transfer finished. Verify your local MP4/ZIP first, then use the optional cell below.


In [ ]:
#@title 6. Optional: disconnect after you confirm downloads
DOWNLOADS_CONFIRMED_SAVED = False #@param {type:"boolean"}
if DOWNLOADS_CONFIRMED_SAVED:
    from google.colab import runtime
    runtime.unassign()
else:
    print("Runtime retained. Confirm the local downloads, then enable the checkbox to release it.")
